In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
TEMP_DIR  = Path("GeoFiles/Temperature_800m/FarmWise")
PREC_DIR  = Path("GeoFiles/Precipitation_800m/FarmWise")
ET_DIR    = Path("GeoFiles/ET/FarmWise")
YIELD_DIR = Path("GeoFiles/Farm_Yield")
SOIL_PATH = Path("GeoFiles/Soil/FarmWise/Farm_SoilTaxonomy_clean.shp")

YEARS = range(2019, 2025) 
PERCENTILE = 5

In [ ]:
# --- Load soil data ---
gdf_soil = gpd.read_file(SOIL_PATH)
print(f"CRS Soil: {gdf_soil.crs}")
gdf_soil = gdf_soil[["CSBID", "SoilClass"]]

# DROP DUPLICATES to ensure pure 1-to-1 merge
gdf_soil = gdf_soil.drop_duplicates(subset=["CSBID"])

gdf_soil["SoilClass"] = gdf_soil["SoilClass"].astype("Int64")  # keep as integer codes
print(f"Loaded soil taxonomy for {len(gdf_soil)} farms")

In [ ]:
#initialize an empty set to keep track of all unique farms across years
all_farms = set()

for year in YEARS:
    print(f"\n📅 Processing year {year}...")

    temp_path = TEMP_DIR / f"Farm_Temperature_{year}.shp"
    prec_path = PREC_DIR / f"Farm_Precipitation_{year}.shp"
    et_path   = ET_DIR   / f"Farm_ETmm_{year}.shp"
    yield_path= YIELD_DIR / f"AZ_Yield_{year}.shp"

    #print crs of each file
    print(f"  CRS Temp: {gpd.read_file(temp_path).crs}")
    print(f"  CRS Preci: {gpd.read_file(prec_path).crs}")
    print(f"  CRS ET: {gpd.read_file(et_path).crs}")
    print(f"  CRS Yield: {gpd.read_file(yield_path).crs}")

    # ID check
    temp_ids  = set(gpd.read_file(temp_path)["CSBID"])
    prec_ids  = set(gpd.read_file(prec_path)["CSBID"])
    et_ids    = set(gpd.read_file(et_path)["CSBID"])
    yield_ids = set(gpd.read_file(yield_path)["CSBID"])
    common_ids = temp_ids & prec_ids & et_ids & yield_ids
    print(f"  Unique farms in Temp: {len(temp_ids)}, Preci: {len(prec_ids)}, "
          f"ET: {len(et_ids)}, Yield: {len(yield_ids)}")
    print(f"  Common farms across all datasets: {len(common_ids)}")

    # Load data (and DROP DUPLICATES!)
    gdf_temp  = gpd.read_file(temp_path)[["CSBID","CNTY","CSBACRES","geometry", f"Temp_{year}",f"TempC_{year}"]].drop_duplicates(subset=["CSBID"])
    gdf_prec  = gpd.read_file(prec_path)[["CSBID", f"Preci_{year}",f"PrecC_{year}"]].drop_duplicates(subset=["CSBID"])
    gdf_et    = gpd.read_file(et_path)[["CSBID", f"ETmm_{year}"]].drop_duplicates(subset=["CSBID"])
    gdf_yield = gpd.read_file(yield_path)[['CSBID','CDL','PredYld_ar','fb_bad','fb_good','Category','Crop']].drop_duplicates(subset=["CSBID"])

    # Merge
    gdf_year = gdf_temp.merge(gdf_prec, on="CSBID", how="inner") \
                       .merge(gdf_et,   on="CSBID", how="inner") \
                       .merge(gdf_yield, on="CSBID", how="inner") \
                       .merge(gdf_soil, on="CSBID", how="inner")

    # Clean categorical fields
    gdf_year["SoilClass"] = gdf_year["SoilClass"].astype("Int64")
    gdf_year["Category"] = gdf_year["Category"].astype(str).str.strip().str.title()
    gdf_year[f"TempC_{year}"] = gdf_year[f"TempC_{year}"].astype(str).str.strip().str.title()
    gdf_year[f"PrecC_{year}"] = gdf_year[f"PrecC_{year}"].astype(str).str.strip().str.title()

    # Build ET thresholds dictionary (all combos, GA-only for percentile)
    et_thresholds = {}
    all_combos = gdf_year[["CDL", f"TempC_{year}", f"PrecC_{year}", "SoilClass"]].drop_duplicates() # all unique combos

    for _, combo in all_combos.iterrows(): # for each unique combo
        key = (combo["CDL"], combo[f"TempC_{year}"], combo[f"PrecC_{year}"], combo["SoilClass"]) # tuple key
        # subset to Good/Average only
        subset = gdf_year[
            (gdf_year["CDL"] == combo["CDL"]) &
            (gdf_year[f"TempC_{year}"] == combo[f"TempC_{year}"]) &
            (gdf_year[f"PrecC_{year}"] == combo[f"PrecC_{year}"]) &
            (gdf_year["SoilClass"] == combo["SoilClass"]) &
            (gdf_year["Category"].isin(["Good", "Average"]))
        ]
        vals = subset[f"ETmm_{year}"].dropna().values # ET values
        # Calculate percentile or NaN if no values
        et_thresholds[key] = np.nanpercentile(vals, PERCENTILE) if vals.size > 0 else np.nan

    print(f"  Unique condition keys (all combos): {len(et_thresholds)}")
    print(f"  Sample thresholds: {list(et_thresholds.items())[:5]}")

    # Assign AttainableET
    attainable_col = f"AT_{year}"
    gdf_year[attainable_col] = gdf_year.apply(
        lambda r: et_thresholds.get((r["CDL"], r[f"TempC_{year}"], r[f"PrecC_{year}"], r["SoilClass"])),
        axis=1
    )

    # Summary
    print(f"  Assigned {gdf_year[attainable_col].notna().sum()} farms")
    print(f"  Missing {gdf_year[attainable_col].isna().sum()} farms")

    missing_keys = [k for k,v in et_thresholds.items() if pd.isna(v)]
    print(f"  Combos with no Good/Average reference: {len(missing_keys)}")
    if missing_keys:
        print("  Sample missing combos:", missing_keys[:10])

    # Compute Volumes
    area_m2 = gdf_year["CSBACRES"] * 4046.8564224

    et_vol_col = f"ETVol{year}"
    at_vol_col = f"ATVol{year}"
    diff_vol_col = f"VolD{year}"
    flag_col = f"BT{year}"

    gdf_year[et_vol_col] = gdf_year[f"ETmm_{year}"] * area_m2 / 1000.0
    gdf_year[at_vol_col] = gdf_year[attainable_col] * area_m2 / 1000.0
    gdf_year[diff_vol_col] = gdf_year[et_vol_col] - gdf_year[at_vol_col]

    # Flag if actual ET is below threshold
    gdf_year[flag_col] = gdf_year[f"ETmm_{year}"] < gdf_year[attainable_col]

    # Drop rows with NaN in Attainable ET
    before = len(gdf_year)
    gdf_year = gdf_year.dropna(subset=[attainable_col])
    after = len(gdf_year)
    print(f"  Dropped {before - after} farms with NaN {attainable_col}")

    #Retaining only necessary columns and renaming for consistency
    cols = ["CSBID","CNTY","CSBACRES","geometry", f"ETmm_{year}", f"PredYld_ar", 'CDL', 'Category', 'SoilClass', f"AT_{year}", f"ETVol{year}", f"ATVol{year}", f"VolD{year}", f"BT{year}"]
    gdf_year = gdf_year[cols]
    gdf_year.rename(columns={
        f"ETmm_{year}": "ETmm",
        f"AT_{year}": "ATmm",
        f"ETVol{year}": "ETVol",
        f"ATVol{year}": "ATVol",
        f"VolD{year}": "VolD",
        f"BT{year}": "BT"
    }, inplace=True)

    gdf_year["Year"] = year
    print(f"  Final rows for {year}: {len(gdf_year)}")

    # Save results
    out_path = f"Final_Vector/AZ_AT_ETFullTable_{year}_{PERCENTILE}.shp"
    gdf_year.to_file(out_path)
    print(f"  Saved to {out_path}")

print("\n✅ Finished loading, QC’ing, categorizing, and attaching soil taxonomy.")

In [ ]:
#Concat vertically all years into a single GeoDataFrame
gdf_all_years = pd.concat([gpd.read_file(f"Final_Vector/AZ_AT_ETFullTable_{year}_{PERCENTILE}.shp") for year in YEARS], ignore_index=True)
print(f"\nTotal unique farms across all years: {gdf_all_years['CSBID'].nunique()}")
print(f"Total records across all years: {len(gdf_all_years)}")

#Number of records per year
print("\nRecords per year:")
print(gdf_all_years["Year"].value_counts().sort_index())